<a href="https://colab.research.google.com/github/perarneskjelvik/Selvstudium/blob/main/Funksjon_Pluss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Funksjon Pluss

Forfatter: Per Arne Skjelvik

Dato: 19.2.2026

##Formål
Denne notebooken inneholder kode som bruker nevrale nettverk for å lære pluss-funksjonen.


In [ ]:

# importerer nødvendige bibliotek
# !pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.metrics import Recall, Precision, AUC
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, TensorBoard, EarlyStopping
import random
import matplotlib.pyplot as plt


# definerer filstier

logdir = (f'logs/scalars/tensorboard')
tb = TensorBoard(log_dir=logdir)

# TensorBoard er et visualiseringsverktøy som viser hvordan treningen går
# ved å vise/plotte målinger som vi definerer.
# Tensorboard kan lastes både før og etter treningen med følgende:
# %load_ext tensorboard
# %tensorboard --logdir logs
# Tensorboard leser fra en katalog som spesifiseres med --logdir FOLDERNAME
# og leter etter tensorboard logfiler.
# Tensorboard logfiler genereres kun hvis tensorboard callback sendes
# til nettverket når du setter det opp til trening.
# Logfilene inneholder all målingene ved hvert tidspunkt under
# trening og validering.

# For å unngå overskriving hvis vi trener netteverket flere ganger
# må vi inkludere tidspunkt i filstien.


In [ ]:
# Genererer data

# Treningsdata pluss-funksjon
x = []
y = []

# lager en tabell med 10000 treningsdata
# lar x variere mellom 0 og 50
# beregner ikke y for x=10 eller x=20  (4 testdata)

for i in range(0,10000):
    xi = random.randint(0,5000)
    xi = xi/100
    xi2 = random.randint(0,5000)
    xi2 = xi2/100
    if(xi==10.0 or xi==20.0):
      if(xi2==10.0 or xi2==20.0):
        continue
    yi = xi+xi2
    x.append([xi,xi2])
    y.append(yi)




In [ ]:
# Viser neon av dataene.

print(np.array(x)[0:20])



In [ ]:
# Definerer, trener, og validerer modellen.

# Setter seed for random.
# Får da samme beregning av vekter hver gang vi kjører koden.

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


# Definerer lagene/strukturen i modellen

# Nettverket er sekvensielt, dvs hvert lag sender output direkte til neste lag.
model = tf.keras.models.Sequential()

# Definerer input layer som har 1 input
model.add(tf.keras.Input(shape=(2,)))

# Legger til 2 ekstra skjulte lag
model.add(tf.keras.layers.Dense(units=32, activation="relu"))
model.add(tf.keras.layers.Dense(units=32, activation="relu"))


# Definerer output layer som har kun 1 node/output
# Bruker linear som aktiveringsfunksjon .....
model.add(tf.keras.layers.Dense(units=1, activation='linear'))

print(np.array(x).shape)

print("Oppsummering modell:")
print("")
print(model.summary())
print("")


# Plotter modellen til en *.jpg fil
plot_model(model, to_file='plottfil.jpg', show_shapes=True)

# Kompilerer modellen
# Angir hvordan modellen skal oppdatere seg selv for å lære.
# Optimizer: Algoritmen som justerer vektene i nettverket
#            Eksempler: adam, sgd, rmsprop.
# Loss Function: Dette er funksjonen modellen prøver å minimere for å måle
#                hvor feil prediksjonene er.
#                Eksempler: binary_crossentropy (for binær klassifisering),
#                           mean_squared_error (for regresjon).
# Metrics (Måltall): Brukes for å vurdere ytelsen til modellen under trening
#                    og testing
#                    Disse påvirker ikke selve oppdateringen av vektene,
#                    men lar deg se hvor godt modellen fungerer.

model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.MeanSquaredError(),
              metrics=['accuracy', Recall(name='recall'),
                       Precision(name='precision'), AUC(name='auc')])


# Trener modellen
model.fit(np.array(x), np.array(y), batch_size=256, epochs=50, callbacks=tb)

# lagrer modellen
model.save('funksjon_pluss.keras')



In [ ]:
# Laster modellen som er lagret (bare for å teste loading)
from tensorflow.keras.models import load_model
loaded_model = load_model('funksjon_pluss.keras')

# Tester modellen
print("Tester modellen:")
q = loaded_model.predict( np.array( [[10,10],[10,20],[20,20],[20,10]] )  )
print(q)

# Fasit
print("Fasit:")
print("20")
print("30")
print("40")
print("30")


original_values_for_plot = [x + 25 for x in range(0, 50)]

plt.figure()
plt.plot(original_values_for_plot, label='Fasit (x+25)')
predicted_input = np.array([[x, 25] for x in range(0, 50)]) # Create pairs like [x, 25]
predicted = loaded_model.predict(predicted_input)

#plt.figure()
plt.plot(predicted, label='Forslag')
plt.legend()
plt.show()

# Starter tensorboard :

%load_ext tensorboard
%tensorboard --logdir logs

